# NILM pipeline — train, so sánh cấu hình detector, export weights

**Pipeline**
- 6 thiết bị: Fridge, WashMach, Dishwash, Microwave, Toaster, Kettle (TV / HiFi / Fan → "Other").
- **Detector** (Conv1d + BiGRU) → P(ON). **Regressor** (NILMFormer) → công suất "nếu đang ON" (loss chỉ ở bước ON).
- Train **cùng lúc + chặn gradient**: detector học `loss phân loại + λ·phản biện(P(ON) × stopgrad(công suất), thực tế)`, λ = 5.
- Suy luận: P(ON) = trung bình detector, công suất = trung vị regressor, **mặt nạ mềm** `P(ON) × công suất`, Other = tổng − thiết bị.

**Vòng này: sửa detector (P(ON) chưa tách biệt bật/tắt)**
1. **Focal loss**: BCE có trọng số `pos_weight` như cũ × `(1 − p_t)^γ` → tập trung vào bước khó.
2. **Cửa sổ dài hơn**: 128 → 256 / 512 bước (32 phút → 64 / 128 phút) để thấy trọn chu trình máy giặt — tạo bằng `python dataset/training/data.py --window 256 512`.

**Quy trình**
1. Chia theo thời gian: train | 1 ngày đệm | val | test (20% cuối).
2. **Giai đoạn A** — mỗi cấu hình: train trên train, dừng sớm trên val. So sánh trên **val**, cùng một tập mốc thời gian.
   **Quy tắc chọn (cố định trước):** F1 năng lượng (macro) cao nhất trên val; hoà thì sai số kWh nhỏ hơn.
3. **Giai đoạn B** — cấu hình được chọn: train lại trên toàn bộ dữ liệu trước test, số epoch = trung vị giai đoạn A.
4. Đánh giá **test một lần**. 5. Export **weights + config**.

> Chạy notebook với thư mục làm việc là `dataset/training`.

In [1]:
import copy
import sys
from dataclasses import asdict
from pathlib import Path
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import clear_output

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

TRAINING_DIR = Path.cwd().resolve()                     # dataset/training
assert (TRAINING_DIR / "train.ipynb").exists(), "Mở notebook với cwd = dataset/training"
REPO_ROOT = TRAINING_DIR.parents[1]
if str(REPO_ROOT) not in sys.path:                      # phải đứng TRƯỚC import app.ml.nilm
    sys.path.insert(0, str(REPO_ROOT))

from app.ml.nilm.GRUDetector import GRUConfig, GRUDetector
from app.ml.nilm.NILMFormerRegressor import NILMFormerConfig, NILMFormerRegressor

DATA_FILE = TRAINING_DIR / "data" / "refit_house2_windows.npz"
CKPT_DIR  = TRAINING_DIR / "checkpoints"
CKPT_DIR.mkdir(exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch {torch.__version__} | device: {DEVICE}")

torch 2.15.0+cu132 | device: cuda


## 1. Dữ liệu (cửa sổ gốc 128 bước × 15 s)

In [2]:
with np.load(DATA_FILE) as z:
    aggregate       = z["aggregate"]                                # (N, L)     W
    time_features   = z["time_features"]                            # (N, 8, L)  sin/cos phút, giờ, thứ, tháng
    targets         = z["targets"]                                  # (N, 9, L)  W, cả 9 kênh Appliance
    window_start    = pd.to_datetime(z["window_start"])             # (N,)
    appliance_names = z["appliance_names"].astype(str).tolist()
    SAMPLE_SECONDS  = int(z["sample_seconds"])

BASE_L = aggregate.shape[-1]
assert window_start.is_monotonic_increasing
print(aggregate.shape, targets.shape, time_features.shape)
print(window_start.min(), "->", window_start.max())

(18328, 128) (18328, 9, 128) (18328, 8, 128)
2013-09-17 22:13:15 -> 2015-05-28 07:26:30


In [3]:
core_channels = ["Appliance1", "Appliance2", "Appliance3", "Appliance5", "Appliance6", "Appliance8"]
labels = {
    "Appliance1": "Fridge",
    "Appliance2": "WashMach",
    "Appliance3": "Dishwash",
    "Appliance5": "Microwave",
    "Appliance6": "Toaster",
    "Appliance8": "Kettle",
}
names = [labels[n] for n in core_channels]
core_idx = [appliance_names.index(n) for n in core_channels]

x_base = np.concatenate([aggregate[:, None], time_features], axis=1).astype(np.float32)  # (N, 9, 128)
y_base = np.ascontiguousarray(targets[:, core_idx], dtype=np.float32)                    # (N, 6, 128) W

# Nhãn ON/OFF theo ngưỡng chuyên gia (W)
on_threshold_w = {"Appliance1": 50, "Appliance2": 20, "Appliance3": 10,
                  "Appliance5": 200, "Appliance6": 200, "Appliance8": 1000}
thresholds_w = np.array([on_threshold_w[n] for n in core_channels], dtype=np.float32)
on_base = y_base >= thresholds_w[None, :, None]                                          # (N, 6, 128) bool
print(x_base.shape, y_base.shape, names)

(18328, 9, 128) (18328, 6, 128) ['Fridge', 'WashMach', 'Dishwash', 'Microwave', 'Toaster', 'Kettle']


## 2. Chia dữ liệu theo thời gian

Val là giai đoạn liền trước test, cách train 1 ngày (tránh rò rỉ giữa các cửa sổ kề nhau của cùng một chu trình). Test chỉ dùng **một lần** ở cuối. Mỗi cửa sổ gốc được gán nhãn `train / gap / val / test`.

In [4]:
TEST_FRACTION, VAL_FRACTION, VAL_GAP = 0.2, 0.15, pd.Timedelta(days=1)

t0, t1     = window_start.min(), window_start.max()
test_start = t0 + (t1 - t0) * (1 - TEST_FRACTION)
pre        = window_start < test_start
tp         = window_start[pre]
val_start  = tp.min() + (tp.max() - tp.min()) * (1 - VAL_FRACTION)

split_label = np.where(~pre, "test",
              np.where(window_start >= val_start, "val",
              np.where(window_start < val_start - VAL_GAP, "train", "gap")))
for s in ["train", "gap", "val", "test"]:
    m = split_label == s
    print(f"{s:5s} {m.sum():6,} windows  {window_start[m].min()} .. {window_start[m].max()}")

train 10,635 windows  2013-09-17 22:13:15 .. 2014-11-10 17:31:15
gap       43 windows  2014-11-10 18:28:45 .. 2014-11-11 17:25:45
val    3,042 windows  2014-11-11 17:57:45 .. 2015-01-24 19:38:15
test   4,608 windows  2015-01-24 20:10:15 .. 2015-05-28 07:26:30


## 3. Cửa sổ dài (256 / 512 bước) từ `data.py`

`python dataset/training/data.py --window 256 512` cắt cửa sổ dài trực tiếp từ chuỗi 15 s (cùng gốc với cửa sổ 128), nên cửa sổ dài `128·k` phủ đúng `k` cửa sổ 128 liền kề. `members` giữ chỉ số các cửa sổ 128 đó → đánh giá mọi độ dài trên **cùng các mốc thời gian** và gán split. Cửa sổ vắt qua ranh giới split bị loại khỏi train/val/test.

In [5]:
DATA_DIR = DATA_FILE.parent
STEP     = pd.Timedelta(seconds=SAMPLE_SECONDS * BASE_L)
base_pos = pd.Series(np.arange(len(window_start)), index=window_start)


def load_windows(L: int) -> SimpleNamespace:
    """Cửa sổ dài L (bội số của 128) từ data.py, kèm chỉ số cửa sổ gốc và nhãn split."""
    path = DATA_FILE if L == BASE_L else DATA_DIR / f"refit_house2_windows_L{L}.npz"
    if not path.exists():
        raise FileNotFoundError(f"{path.name} chưa có: chạy  python dataset/training/data.py --window {L}")
    with np.load(path) as z:
        agg, tf, tg = z["aggregate"], z["time_features"], z["targets"]
        starts = pd.to_datetime(z["window_start"])
        assert z["appliance_names"].astype(str).tolist() == appliance_names
    k = L // BASE_L
    members = np.stack([base_pos.reindex(starts + j * STEP).to_numpy() for j in range(k)], axis=1)
    assert not np.isnan(members).any(), "cửa sổ dài không khớp lưới cửa sổ 128"
    members = members.astype(np.int64)                                                # (N_L, k)
    lab = split_label[members]
    x = np.concatenate([agg[:, None], tf], axis=1).astype(np.float32)
    y = np.ascontiguousarray(tg[:, core_idx], dtype=np.float32)
    return SimpleNamespace(k=k, L=L, members=members, x=x, y=y, on=y >= thresholds_w[None, :, None],
                           label=np.where((lab == lab[:, :1]).all(1), lab[:, 0], None),   # None: vắt qua ranh giới
                           pretest=(lab != "test").all(1))                                # dùng cho giai đoạn B


def to_base(D, values: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """(n, C, 128·k) theo cửa sổ dài -> (n·k, C, 128) theo cửa sổ gốc + chỉ số cửa sổ gốc."""
    n, C, _ = values.shape
    return values.reshape(n, C, D.k, BASE_L).transpose(0, 2, 1, 3).reshape(n * D.k, C, BASE_L), D.members.ravel()


for L in (128, 256, 512):
    D = load_windows(L)
    cov = {s: (D.label == s).sum() * D.k / (split_label == s).sum() for s in ("train", "val", "test")}
    print(f"L={L:4d}: {len(D.members):6,} cửa sổ | độ phủ " + " ".join(f"{s}={100 * v:.0f}%" for s, v in cov.items()))

L= 128: 18,328 cửa sổ | độ phủ train=100% val=100% test=100%
L= 256:  8,407 cửa sổ | độ phủ train=88% val=96% test=97%
L= 512:  3,631 cửa sổ | độ phủ train=71% val=90% test=91%


## 4. Cấu hình và các hàm dùng chung

- `detector_inputs` / `regressor_inputs`: **nguồn duy nhất** của tiền xử lý (train, đánh giá, export; app làm y hệt).
- Loss phân loại của detector: `"bce"` = BCE với `pos_weight = sqrt(#OFF/#ON)` (≤ 30); `"focal"` = **cùng BCE đó × (1 − p_t)^γ** → khác biệt duy nhất là tập trung vào bước khó.

In [6]:
SEEDS          = [11, 22, 33, 44, 55]
LAMBDA         = 5.0           # trọng số phản biện của regressor trong loss detector
FOCAL_GAMMA    = 2.0
EPOCHS         = 50            # tối đa ở giai đoạn A
PATIENCE       = 7
BASE_BATCH     = 64            # batch cho cửa sổ 128; cửa sổ dài: 64 // k (giữ số bước thời gian / lần cập nhật)
REG_LR, DET_LR = 1e-4, 1e-3
POS_WEIGHT_MAX = 30.0
TAU_W          = 10_000.0      # regressor nhìn aggregate / 10 kW

gru_config        = GRUConfig(c_in=x_base.shape[1] + 1, c_out=len(core_channels))    # kW + bước nhảy + 8 time features
nilmformer_config = NILMFormerConfig(c_out=len(core_channels))
LOSS_NAMES        = ["reg", "cls", "crit"]

# Cấu hình so sánh ở giai đoạn A: tên -> (độ dài cửa sổ, loss phân loại)
EXPERIMENTS = {
    "bce_L128":   (128, "bce"),     # hiện tại
    "focal_L128": (128, "focal"),
    "focal_L256": (256, "focal"),
    "focal_L512": (512, "focal"),
}


def detector_inputs(xb: torch.Tensor) -> torch.Tensor:
    """(B, 9, L) [aggregate W, time features] -> (B, 10, L) [aggregate kW, bước nhảy kW, time features]."""
    agg_kw  = xb[:, :1] / 1000.0
    diff_kw = torch.diff(agg_kw, dim=-1, prepend=agg_kw[..., :1])
    return torch.cat([agg_kw, diff_kw, xb[:, 1:]], dim=1)


def regressor_inputs(xb: torch.Tensor) -> torch.Tensor:
    """(B, 9, L) -> (B, 9, L) [aggregate / TAU_W, time features]."""
    return torch.cat([xb[:, :1] / TAU_W, xb[:, 1:]], dim=1)


def channel_scale(D, ids: np.ndarray) -> np.ndarray:
    """Chuẩn hoá target theo từng kênh: std (W) trên tập fit; 0 W vẫn là 0."""
    return np.maximum(D.y[ids].std(axis=(0, 2)), 1e-6).astype(np.float32)


def positive_weight(D, ids: np.ndarray) -> torch.Tensor:
    """pos_weight: sqrt(#OFF / #ON), kẹp trong [1, POS_WEIGHT_MAX]."""
    p = D.on[ids].mean(axis=(0, 2))
    w = np.clip(np.sqrt((1 - p) / np.maximum(p, 1e-6)), 1, POS_WEIGHT_MAX)
    return torch.as_tensor(w, dtype=torch.float32, device=DEVICE)[:, None]        # (C, 1)


def make_loader(D, ids: np.ndarray, shuffle: bool, seed: int) -> DataLoader:
    ds = TensorDataset(torch.from_numpy(D.x[ids]), torch.from_numpy(D.y[ids]),
                       torch.from_numpy(D.on[ids].astype(np.float32)))
    gen = torch.Generator().manual_seed(seed) if shuffle else None
    return DataLoader(ds, batch_size=max(BASE_BATCH // D.k, 8), shuffle=shuffle, generator=gen)


def classification_loss(logits, ob, pos_weight, det_loss: str) -> torch.Tensor:
    """BCE có trọng số (pos_weight cho bước ON); focal = BCE đó × (1 − p_t)^γ."""
    bce = F.binary_cross_entropy_with_logits(logits, ob, reduction="none")
    weight = 1 + (pos_weight - 1) * ob                                            # pos_weight ở bước ON, 1 ở bước OFF
    if det_loss == "focal":
        p = torch.sigmoid(logits)
        p_t = p * ob + (1 - p) * (1 - ob)                                         # xác suất của nhãn đúng
        weight = weight * (1 - p_t).pow(FOCAL_GAMMA)                              # bước dễ (p_t ~ 1) gần như bị bỏ qua
    return (weight * bce).mean()


def compute_losses(regressor, detector, batch, scale_w, pos_weight, det_loss):
    """[reg, cls, crit] (đơn vị chuẩn hoá W / scale_w) và bảng đếm ON/OFF (C, 4)."""
    xb, yb_w, ob = (b.to(DEVICE) for b in batch)
    yb = yb_w / scale_w                                                            # (B, C, L)

    y_hat  = regressor(regressor_inputs(xb))                                       # công suất "nếu ON"
    logits = detector(detector_inputs(xb))                                         # ON/OFF

    loss_reg  = (ob * (y_hat - yb) ** 2).sum() / ob.sum().clamp(min=1)            # chỉ ở bước ON
    loss_cls  = classification_loss(logits, ob, pos_weight, det_loss)
    gated     = torch.sigmoid(logits) * y_hat.detach().clamp(min=0)               # chặn gradient vào regressor
    loss_crit = F.mse_loss(gated, yb) + F.mse_loss(gated.mean(-1), yb.mean(-1))   # từng bước + năng lượng cửa sổ

    # Bảng đếm ON/OFF ở ngưỡng 0.5 (logit > 0), chỉ để theo dõi
    pred_on, true_on = logits.detach() > 0, ob > 0
    counts = torch.stack([(pred_on & true_on).sum((0, 2)), (pred_on & ~true_on).sum((0, 2)),
                          (~pred_on & true_on).sum((0, 2)), (~pred_on & ~true_on).sum((0, 2))], dim=1)  # (C, 4) tp fp fn tn
    return torch.stack([loss_reg, loss_cls, loss_crit]), counts


def detection_scores(counts: torch.Tensor) -> tuple[float, float]:
    """(C, 4) [tp, fp, fn, tn] -> (accuracy, F1), trung bình macro trên các thiết bị."""
    tp, fp, fn, tn = counts.double().unbind(1)
    accuracy = ((tp + tn) / (tp + fp + fn + tn).clamp(min=1)).mean()
    f1 = (2 * tp / (2 * tp + fp + fn).clamp(min=1)).mean()
    return float(accuracy), float(f1)

In [7]:
def plot_history(history: dict, best_epoch: int, title: str) -> None:
    clear_output(wait=True)
    ep = range(1, len(history["train_reg"]) + 1)
    has_val = len(history["val_reg"]) > 0
    fig, (ax_det, ax_reg) = plt.subplots(1, 2, figsize=(15, 4.5))

    # Detector: accuracy và F1 (macro, ngưỡng 0.5)
    for key, c in [("acc", "C0"), ("f1", "C2")]:
        label = "accuracy" if key == "acc" else "F1"
        ax_det.plot(ep, history[f"train_{key}"], f"{c}-", label=f"train {label}")
        if has_val:
            ax_det.plot(ep, history[f"val_{key}"], f"{c}--", label=f"val {label}")
    ax_det.set_ylim(0, 1)
    ax_det.set_ylabel("score (macro, ngưỡng 0.5)")

    # Regressor: loss ON-only
    ax_reg.plot(ep, history["train_reg"], "C3-", label="train reg (ON-only MSE)")
    if has_val:
        ax_reg.plot(ep, history["val_reg"], "C3--", label="val reg (ON-only MSE)")
    ax_reg.set_ylabel("loss")

    for ax, t in [(ax_det, "Detector (GRU)"), (ax_reg, "Regressor (NILMFormer)")]:
        if has_val and best_epoch:
            ax.axvline(best_epoch, color="grey", ls=":", label=f"best epoch {best_epoch}")
        ax.set_xlabel("epoch"); ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
        ax.grid(alpha=0.3); ax.legend(fontsize=8); ax.set_title(t)
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()


def train_seed(D, seed: int, fit_ids: np.ndarray, val_ids: np.ndarray | None, epochs: int,
               det_loss: str, title: str) -> dict:
    """val_ids != None: dừng sớm trên val (tổng loss), giữ epoch tốt nhất (giai đoạn A).
       val_ids == None: train đúng `epochs` epoch, giữ epoch cuối (giai đoạn B)."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    scale_np   = channel_scale(D, fit_ids)
    scale_w    = torch.as_tensor(scale_np, device=DEVICE)[None, :, None]
    pos_weight = positive_weight(D, fit_ids)

    regressor = NILMFormerRegressor(nilmformer_config).to(DEVICE)
    detector  = GRUDetector(gru_config).to(DEVICE)
    optimizers = [torch.optim.AdamW(regressor.parameters(), lr=REG_LR),
                  torch.optim.AdamW(detector.parameters(),  lr=DET_LR)]
    fit_loader = make_loader(D, fit_ids, shuffle=True, seed=seed)
    val_loader = make_loader(D, val_ids, shuffle=False, seed=seed) if val_ids is not None else None

    history = {f"{s}_{n}": [] for s in ("train", "val") for n in LOSS_NAMES + ["acc", "f1"]}
    best    = {"score": float("inf"), "epoch": 0}

    for epoch in range(1, epochs + 1):
        regressor.train(); detector.train()
        sums, counts, n = torch.zeros(3), 0, 0
        for batch in fit_loader:
            losses, c = compute_losses(regressor, detector, batch, scale_w, pos_weight, det_loss)
            for opt in optimizers:
                opt.zero_grad(set_to_none=True)
            (losses[0] + losses[1] + LAMBDA * losses[2]).backward()   # gradient tách rời nhờ detach
            for opt in optimizers:
                opt.step()
            sums += losses.detach().cpu(); counts = counts + c.cpu(); n += 1
        for name, v in zip(LOSS_NAMES, (sums / n).tolist()):
            history[f"train_{name}"].append(v)
        acc, f1 = detection_scores(counts)
        history["train_acc"].append(acc); history["train_f1"].append(f1)

        stop = False
        if val_loader is not None:
            regressor.eval(); detector.eval()
            with torch.no_grad():
                outs = [compute_losses(regressor, detector, b, scale_w, pos_weight, det_loss) for b in val_loader]
            val = torch.stack([o[0] for o in outs]).mean(0).cpu().numpy()
            val_acc, val_f1 = detection_scores(sum(o[1].cpu() for o in outs))
            for name, v in zip(LOSS_NAMES, val.tolist()):
                history[f"val_{name}"].append(v)
            history["val_acc"].append(val_acc); history["val_f1"].append(val_f1)

            score = float(val.sum())                                   # dừng sớm theo tổng loss val
            if score < best["score"]:
                best = {"score": score, "epoch": epoch,
                        "regressor": copy.deepcopy(regressor.state_dict()),
                        "detector":  copy.deepcopy(detector.state_dict())}
            stop = epoch - best["epoch"] >= PATIENCE

        plot_history(history, best["epoch"], f"{title} — epoch {epoch}/{epochs}")
        if val_loader is not None:
            print(f"seed={seed} epoch={epoch:2d} val " +
                  " ".join(f"{k}={v:.4f}" for k, v in zip(LOSS_NAMES, val)) +
                  f" score={score:.4f} | acc={val_acc:.3f} F1={val_f1:.3f} | best epoch {best['epoch']}")
        else:
            print(f"seed={seed} epoch={epoch:2d} train acc={acc:.3f} F1={f1:.3f}")
        if stop:
            break

    if val_loader is None:          # giai đoạn B: giữ epoch cuối
        best = {"score": float("nan"), "epoch": epoch,
                "regressor": regressor.state_dict(), "detector": detector.state_dict()}

    return {
        "seed": seed, "lambda": LAMBDA, "det_loss": det_loss, "focal_gamma": FOCAL_GAMMA,
        "window": D.L, "epochs_trained": epoch,
        "best_epoch": best["epoch"], "best_val_score": best["score"],
        "tau_w": TAU_W, "scale_w": scale_np.tolist(),
        "detector_config":  asdict(gru_config),
        "regressor_config": asdict(nilmformer_config),
        "detector_state":  {k: v.cpu() for k, v in best["detector"].items()},
        "regressor_state": {k: v.cpu() for k, v in best["regressor"].items()},
        "history": history,
    }

## 5. Giai đoạn A — train từng cấu hình, chọn epoch trên val

Mỗi cấu hình × 5 seed. **Không nhìn test.**

In [8]:
windows = {L: load_windows(L) for L in sorted({L for L, _ in EXPERIMENTS.values()})}
select  = {}                                    # select[name][seed] = checkpoint

for name, (L, det_loss) in EXPERIMENTS.items():
    D = windows[L]
    train_ids, val_ids = np.flatnonzero(D.label == "train"), np.flatnonzero(D.label == "val")
    select[name] = {}
    for seed in SEEDS:
        ck = train_seed(D, seed, train_ids, val_ids, EPOCHS, det_loss,
                        f"Giai đoạn A — {name} — seed {seed} ({SEEDS.index(seed) + 1}/{len(SEEDS)})")
        torch.save(ck, CKPT_DIR / f"select_{name}_seed_{seed}.pt")
        select[name][seed] = ck

clear_output()
print(pd.DataFrame({name: {s: select[name][s]["best_epoch"] for s in SEEDS} for name in EXPERIMENTS}).rename_axis("seed"))

      bce_L128  focal_L128  focal_L256  focal_L512
seed                                              
11          14          14          22          23
22          31          27          15          16
33          21          18          13          18
44          20          20          17          23
55          11          11          18          17


## 6. Ensemble, suy luận và các thước đo

`predict` là **toàn bộ logic suy luận** (app tái hiện đúng hàm này): P(ON) = trung bình detector; công suất = trung vị regressor (×`scale_w`, ≥ 0); thiết bị = P(ON) × công suất; Other = tổng − thiết bị.

Thước đo cho vấn đề "P(ON) chưa tách biệt" (tính trên cửa sổ gốc, macro 6 thiết bị):
- **Năng lượng khớp** = Σ min(dự đoán, thực tế) theo từng bước → *precision* = khớp / Σ dự đoán, *recall* = khớp / Σ thực tế, *F1* = 2·khớp / (Σ dự đoán + Σ thực tế). Năng lượng đặt sai thời điểm **không** được tính.
- **Độ tách biệt** = P(ON) trung bình khi ON − khi OFF.
- Sai số kWh mỗi thiết bị, F1 ON/OFF (ngưỡng chỉnh trên val).

In [9]:
def build_members(members: list[dict]):
    """members: list các dict {detector_config, regressor_config, detector_state, regressor_state, scale_w}."""
    detectors, regressors, scales = [], [], []
    for m in members:
        d = GRUDetector(GRUConfig(**m["detector_config"]));                 d.load_state_dict(m["detector_state"])
        r = NILMFormerRegressor(NILMFormerConfig(**m["regressor_config"])); r.load_state_dict(m["regressor_state"])
        detectors.append(d.to(DEVICE).eval()); regressors.append(r.to(DEVICE).eval())
        scales.append(torch.as_tensor(m["scale_w"], dtype=torch.float32, device=DEVICE))
    return detectors, regressors, torch.stack(scales)                       # scales: (M, C)


@torch.no_grad()
def predict(ensemble, xb: torch.Tensor, gate: str = "soft", event_thresholds=None):
    """xb: (B, 9, L) [aggregate W, time features] -> p_on, power_w, appliance_w (B, C, L), other_w (B, L)."""
    detectors, regressors, scales = ensemble
    z, xr = detector_inputs(xb), regressor_inputs(xb)
    p_on    = torch.stack([torch.sigmoid(d(z)) for d in detectors]).mean(0)
    power_w = torch.stack([r(xr) * s[None, :, None] for r, s in zip(regressors, scales)]).median(0).values.clamp(min=0)
    g = p_on if gate == "soft" else (p_on >= event_thresholds[None, :, None]).float()
    appliance_w = power_w * g
    other_w = (xb[:, 0] - appliance_w.sum(1)).clamp(min=0)
    return p_on, power_w, appliance_w, other_w


def run_base(ensemble, D, ids: np.ndarray, batch: int = 64) -> dict:
    """Suy luận trên cửa sổ dài rồi trả về theo cửa sổ gốc: {base_ids, p_on, appliance_w} (n·k, C, 128)."""
    p, a = [], []
    for i in range(0, len(ids), batch):
        o = predict(ensemble, torch.from_numpy(D.x[ids[i:i + batch]]).to(DEVICE))
        p.append(o[0].cpu().numpy()); a.append(o[2].cpu().numpy())
    sub = SimpleNamespace(k=D.k, members=D.members[ids])
    p_on, base_ids = to_base(sub, np.concatenate(p))
    app_w, _ = to_base(sub, np.concatenate(a))
    return {"base_ids": base_ids, "p_on": p_on, "appliance_w": app_w}


def restrict(out: dict, common: np.ndarray) -> dict:
    pos = {b: i for i, b in enumerate(out["base_ids"])}
    sel = np.array([pos[b] for b in common])
    return {"p_on": out["p_on"][sel], "appliance_w": out["appliance_w"][sel]}


THRESHOLD_GRID = np.linspace(0.02, 0.98, 49)


def onoff_f1(t: np.ndarray, p: np.ndarray) -> float:
    tp, fp, fn = (t & p).sum(), (~t & p).sum(), (t & ~p).sum()
    return 2 * tp / max(2 * tp + fp + fn, 1)


def tune_event_thresholds(p_on: np.ndarray, true_on: np.ndarray) -> np.ndarray:
    return np.array([THRESHOLD_GRID[int(np.argmax([onoff_f1(true_on[:, c], p_on[:, c] >= t) for t in THRESHOLD_GRID]))]
                     for c in range(true_on.shape[1])], dtype=np.float32)


def evaluate(out: dict, common: np.ndarray, event_thresholds: np.ndarray):
    """Bảng theo thiết bị + tóm tắt macro, trên các cửa sổ gốc `common`."""
    pred, p_on = out["appliance_w"], out["p_on"]
    true, true_on = y_base[common], on_base[common]
    k_ = SAMPLE_SECONDS / 3.6e6
    rows = []
    for c, n in enumerate(names):
        t_, p_ = true[:, c], pred[:, c]
        match = np.minimum(t_, p_).sum()
        on_c = true_on[:, c]
        rows.append({"true kWh": t_.sum() * k_, "pred kWh": p_.sum() * k_,
                     "kWh err %": 100 * (p_.sum() / t_.sum() - 1),
                     "energy P": match / max(p_.sum(), 1e-9), "energy R": match / max(t_.sum(), 1e-9),
                     "energy F1": 2 * match / max(p_.sum() + t_.sum(), 1e-9),
                     "P(ON) khi ON": p_on[:, c][on_c].mean(), "P(ON) khi OFF": p_on[:, c][~on_c].mean(),
                     "F1 ON/OFF": onoff_f1(on_c, p_on[:, c] >= event_thresholds[c])})
    table = pd.DataFrame(rows, index=names)
    table["tách biệt"] = table["P(ON) khi ON"] - table["P(ON) khi OFF"]
    summary = {"energy F1": table["energy F1"].mean(), "energy P": table["energy P"].mean(),
               "energy R": table["energy R"].mean(), "tách biệt": table["tách biệt"].mean(),
               "F1 ON/OFF": table["F1 ON/OFF"].mean(), "mean |kWh err| %": table["kWh err %"].abs().mean(),
               "total kWh err %": 100 * (table["pred kWh"].sum() / table["true kWh"].sum() - 1)}
    return table, summary

## 7. So sánh trên val và chọn cấu hình

Mọi cấu hình được chấm trên **cùng tập cửa sổ gốc** (giao các vùng mà mọi độ dài cửa sổ đều phủ). Ngưỡng ON/OFF (chỉ cho hiển thị sự kiện và F1 ON/OFF) chỉnh trên chính val của từng cấu hình.

In [10]:
ensembles_A = {name: build_members([select[name][s] for s in SEEDS]) for name in EXPERIMENTS}
val_out = {name: run_base(ensembles_A[name], windows[L], np.flatnonzero(windows[L].label == "val"))
           for name, (L, _) in EXPERIMENTS.items()}
common_val = np.sort(list(set.intersection(*(set(o["base_ids"].tolist()) for o in val_out.values()))))
print(f"so sánh trên {len(common_val):,} cửa sổ gốc của val")

event_thr, val_tables, val_summary = {}, {}, {}
for name in EXPERIMENTS:
    out = restrict(val_out[name], common_val)
    event_thr[name] = tune_event_thresholds(out["p_on"], on_base[common_val])
    val_tables[name], val_summary[name] = evaluate(out, common_val, event_thr[name])

pd.set_option("display.width", 220)
summary_val = pd.DataFrame(val_summary).T
print("=== VAL — tóm tắt (macro) ===\n", summary_val.round(3))
for col in ["energy F1", "tách biệt", "kWh err %"]:
    print(f"\n=== VAL — {col} theo thiết bị ===\n", pd.DataFrame({n: t[col] for n, t in val_tables.items()}).round(3))

# Quy tắc chọn (cố định trước): energy F1 cao nhất; hoà (< 0.005) thì sai số kWh nhỏ hơn
best_f1 = summary_val["energy F1"].max()
tied    = summary_val[summary_val["energy F1"] >= best_f1 - 0.005]
CHOSEN  = tied["mean |kWh err| %"].idxmin()
CHOSEN_L, CHOSEN_LOSS = EXPERIMENTS[CHOSEN]
print(f"\n→ Chọn: {CHOSEN} (cửa sổ {CHOSEN_L}, loss {CHOSEN_LOSS})")

so sánh trên 2,728 cửa sổ gốc của val
=== VAL — tóm tắt (macro) ===
             energy F1  energy P  energy R  tách biệt  F1 ON/OFF  mean |kWh err| %  total kWh err %
bce_L128        0.496     0.468     0.540      0.416      0.610            30.238           15.494
focal_L128      0.490     0.449     0.550      0.412      0.617            37.681           25.289
focal_L256      0.505     0.489     0.527      0.418      0.617            19.380            7.568
focal_L512      0.493     0.479     0.513      0.417      0.600            12.905            0.831

=== VAL — energy F1 theo thiết bị ===
            bce_L128  focal_L128  focal_L256  focal_L512
Fridge        0.541       0.551       0.554       0.568
WashMach      0.190       0.173       0.241       0.229
Dishwash      0.627       0.610       0.611       0.624
Microwave     0.367       0.378       0.379       0.379
Toaster       0.446       0.417       0.456       0.362
Kettle        0.808       0.807       0.786       0.795

===

## 8. Giai đoạn B — train production cho cấu hình đã chọn

Toàn bộ dữ liệu trước test (train + đệm + val), số epoch = trung vị epoch tốt nhất của cấu hình đó ở giai đoạn A.

In [11]:
FINAL_EPOCHS = int(np.median([select[CHOSEN][s]["best_epoch"] for s in SEEDS]))
D_pool       = windows[CHOSEN_L]                                  # cửa sổ có thể vắt qua train/val, không qua test
pool_ids     = np.flatnonzero(D_pool.pretest)

final = {}
for seed in SEEDS:
    ck = train_seed(D_pool, seed, pool_ids, None, FINAL_EPOCHS, CHOSEN_LOSS,
                    f"Giai đoạn B — {CHOSEN} — seed {seed} ({SEEDS.index(seed) + 1}/{len(SEEDS)})")
    torch.save(ck, CKPT_DIR / f"final_{CHOSEN}_seed_{seed}.pt")
    final[seed] = ck
clear_output()
print(f"{CHOSEN}: đã train {len(final)} seed × {FINAL_EPOCHS} epoch trên {len(pool_ids):,} cửa sổ {D_pool.L}")

focal_L256: đã train 5 seed × 17 epoch trên 6,179 cửa sổ 256


## 9. Test — chấm một lần

So sánh: cấu hình đã chọn (A và B) với cấu hình cũ `bce_L128` (A), trên cùng tập cửa sổ gốc của test. Ngưỡng sự kiện ON/OFF lấy từ val của ensemble A tương ứng.

In [12]:
ens_final = build_members([final[s] for s in SEEDS])
D_test    = windows[CHOSEN_L]
runs = {
    "bce_L128 (A, cũ)":           (ensembles_A["bce_L128"], windows[128], event_thr["bce_L128"]),
    f"{CHOSEN} (A)":              (ensembles_A[CHOSEN], D_test, event_thr[CHOSEN]),
    f"{CHOSEN} (B, production)":  (ens_final, D_test, event_thr[CHOSEN]),
}
test_out = {n: run_base(e, D, np.flatnonzero(D.label == "test")) for n, (e, D, _) in runs.items()}
common_test = np.sort(list(set.intersection(*(set(o["base_ids"].tolist()) for o in test_out.values()))))
print(f"test: {len(common_test):,} cửa sổ gốc chung")

test_tables, test_summary = {}, {}
for n, (_, _, thr) in runs.items():
    test_tables[n], test_summary[n] = evaluate(restrict(test_out[n], common_test), common_test, thr)

print("=== TEST — tóm tắt (macro) ===\n", pd.DataFrame(test_summary).T.round(3))
print("\n=== TEST — kWh theo thiết bị ===\n",
      pd.concat({n: t[["true kWh", "pred kWh", "kWh err %"]] for n, t in test_tables.items()}, axis=1).round(1))
print("\n=== TEST — chi tiết production ===\n", test_tables[f"{CHOSEN} (B, production)"].round(3))

test: 4,456 cửa sổ gốc chung
=== TEST — tóm tắt (macro) ===
                             energy F1  energy P  energy R  tách biệt  F1 ON/OFF  mean |kWh err| %  total kWh err %
bce_L128 (A, cũ)                0.529     0.497     0.570      0.442      0.603            16.465           15.667
focal_L256 (A)                  0.521     0.504     0.545      0.440      0.621            15.365           13.625
focal_L256 (B, production)      0.567     0.551     0.589      0.484      0.660            11.370           11.901

=== TEST — kWh theo thiết bị ===
           bce_L128 (A, cũ)                        focal_L256 (A)                        focal_L256 (B, production)                       
                  true kWh    pred kWh  kWh err %       true kWh    pred kWh  kWh err %                   true kWh    pred kWh  kWh err %
Fridge           78.800003   90.000000  14.300000      78.800003   89.099998  13.000000                  78.800003   81.599998   3.600000
WashMach         36.900002   4

## 10. Export weights + config cho app

File chỉ chứa config (kiểu Python thuần) + tensor → `torch.load(weights_only=True)`; app dựng lại bằng `GRUDetector` / `NILMFormerRegressor` trong `app/ml/nilm/` và tái hiện `predict` ở mục 6, với cửa sổ đầu vào dài `window` bước.

In [13]:
BUNDLE_PATH = CKPT_DIR / "nilm_pipeline.pt"
EVENT_THRESHOLDS = event_thr[CHOSEN]

bundle = {
    "format": "wattprint-nilm/2",
    "appliances": core_channels, "labels": names,
    "on_threshold_w": thresholds_w.tolist(),
    "gate": "soft",
    "event_thresholds": EVENT_THRESHOLDS.tolist(),          # chỉ cho hiển thị ON/OFF
    "tau_w": TAU_W, "sample_seconds": SAMPLE_SECONDS, "window": int(D_pool.L),
    "input": f"(B, 9, {D_pool.L}) float32: aggregate W (resample 15 s, clip >= 0) + 8 sin/cos time features (data.py --window {D_pool.L})",
    "output": "appliance W (B, C, L) = P(ON) x power; Other = aggregate - sum(appliances), clamp >= 0",
    "members": [{k: final[s][k] for k in ("seed", "scale_w", "detector_config", "regressor_config",
                                          "detector_state", "regressor_state")} for s in SEEDS],
    "meta": {"config": CHOSEN, "det_loss": CHOSEN_LOSS, "focal_gamma": FOCAL_GAMMA, "lambda": LAMBDA,
             "final_epochs": FINAL_EPOCHS, "seeds": SEEDS,
             "trained_until": str(window_start[split_label != "test"].max()),
             "test_summary": {k: float(v) for k, v in test_summary[f"{CHOSEN} (B, production)"].items()}},
}
torch.save(bundle, BUNDLE_PATH)

# Kiểm tra: load an toàn rồi dựng lại từ app/ml/nilm/ phải ra đúng kết quả
loaded   = torch.load(BUNDLE_PATH, map_location="cpu", weights_only=True)
ens_back = build_members(loaded["members"])
xb = torch.from_numpy(D_test.x[np.flatnonzero(D_test.label == "test")[:16]]).to(DEVICE)
same = all(torch.allclose(a, b, atol=1e-4) for a, b in zip(predict(ens_final, xb), predict(ens_back, xb)))
assert same, "bundle load lại cho kết quả khác"
print(f"exported {BUNDLE_PATH} ({BUNDLE_PATH.stat().st_size / 1e6:.1f} MB), window={bundle['window']}, weights_only reload OK")

exported /home/ai_team/dat/wattprint-backend/dataset/training/checkpoints/nilm_pipeline.pt (10.4 MB), window=256, weights_only reload OK
